# NB03 — SHACL Validation (Shapes Constraint Language)

**This notebook demonstrates:**

1. The difference between **Node Shape** and **Property Shape**
2. Writing four constraint types: cardinality / range / path / logical combinations (and/or/not)
3. Custom constraints via `sh:sparql` (beyond SHACL Core capabilities)
4. Reading `sh:ValidationReport` and automatically locating violating nodes
5. Using `sh:closed` to lock down the set of allowed properties on an instance
6. When to use SHACL vs OWL — they are **complementary**, not substitutes

## Prerequisites

- Phase A data loaded; `ontology/shapes.ttl` exists
- `pyshacl` installed (`uv pip install pyshacl`)

## SHACL vs OWL — A Key Clarification

| | OWL | SHACL |
|---|---|---|
| Purpose | Defines "what a concept is" (ontology) | Validates "is data compliant" |
| Open/Closed World | Open (not stated = unknown) | Closed (not stated = violation, optional) |
| Error form | "possibly inconsistent" / inferred contradiction | Explicit violation report + node reference |
| Performance | Slow (DL), depends on rules | Single-pass scan, stable |

**Rule of thumb**: validate with SHACL before loading instances; reason with OWL after they are in the store.

## 0. Setup

> 🔧 **Tech**: pyshacl `validate()` entry point + `shacl_graph` keyword (old name `shapes_graph` is deprecated and silently fails — see SPEC §11.19)
> 🎯 **Goal**: Load T-Box + A-Box + shapes as three graphs; define a unified `run()` helper reused throughout
> ✅ **Verify**: Triple counts print normally; all subsequent cells can reuse `run()`
> 📚 **Takeaway**: In pySHACL ≥0.30 the kwarg is `shacl_graph` — using the old name silently drops shapes and returns conforms=True, a dangerous false positive

In [1]:
from pathlib import Path
from rdflib import Graph, Namespace, RDF, URIRef
from pyshacl import validate

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ONTO = PROJECT / "ontology"

# Load T-Box + A-Box + shapes
data = Graph()
for f in ["credit_risk.ttl", "instances/customers.ttl", "instances/applications.ttl"]:
    data.parse(str(ONTO / f), format="turtle")
shapes = Graph()
shapes.parse(str(ONTO / "shapes.ttl"), format="turtle")

print(f"Data:  {len(data)} triples")
print(f"Shapes: {len(shapes)} triples")

CR = Namespace("https://nikko.dev/ontology/credit#")
SH = Namespace("http://www.w3.org/ns/shacl#")


def run(data_graph, shapes_graph=shapes, **kwargs):
    """Validate and return (conforms, report_graph, report_text)."""
    return validate(
        data_graph=data_graph,
        shacl_graph=shapes_graph,   # NB: pySHACL >=0.30 renamed shapes_graph
        inference="rdfs",           # apply RDFS expansion before validating
        advanced=True,              # enable sh:sparql
        debug=False,
        **kwargs,
    )

Data:  925 triples
Shapes: 113 triples


## 1. Sanity Check — Clean Fixture Should Have 0 Violations

The Phase A fixtures are hand-crafted to satisfy every shape.

> 🔧 **Tech**: SHACL `sh:ValidationReport` full validation + `conforms` boolean
> 🎯 **Goal**: Run the clean fixture once to confirm baseline conforms=True
> ✅ **Verify**: `conforms == True`, report is empty
> 📚 **Takeaway**: Always confirm baseline is clean before injecting violations — otherwise violation sources become ambiguous

In [2]:
conforms, _, report = run(data)
print(f"Conforms (clean fixture): {conforms}")
if not conforms:
    print(report[:1500])

Conforms (clean fixture): True

## 2. Deliberate Break #1 — Cardinality Violation

`:DecisionCardinalityShape` restricts each application to at most one decision.
Add two conflicting decisions to App_M01 and observe how SHACL reports the error.

> 🔧 **Tech**: `sh:NodeShape` + `sh:property` + `sh:maxCount` cardinality constraint
> 🎯 **Goal**: Inject two `:hasDecision` triples on the same application to trigger a maxCount=1 violation
> ✅ **Verify**: `conforms=False` + report contains `focusNode=App_M01` + `sourceShape=DecisionCardinalityShape` + `MaxCountConstraintComponent`
> 📚 **Takeaway**: SHACL error reports are precise — focusNode / shape / constraint component triple — far friendlier than OWL's "possibly inconsistent"

In [3]:
bad = Graph()
for t in data: bad.add(t)
bad.add((CR.App_M01, CR.hasDecision, CR.Approve))
bad.add((CR.App_M01, CR.hasDecision, CR.Decline))

conforms, _, report = run(bad)
print(f"Conforms: {conforms}")
print(report[:1200])

Conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in MaxCountConstraintComponent (http://www.w3.org/ns/shacl#MaxCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:maxCount Literal("1", datatype=xsd:integer) ; sh:message Literal("An application cannot have multiple decisions", lang=en) ; sh:path :hasDecision ]
	Focus Node: <https://nikko.dev/ontology/credit#App_M01>
	Result Path: :hasDecision
	Message: An application cannot have multiple decisions



**Observation**: The violation report pinpoints the exact *focusNode* (`App_M01`), the violated *Shape* (`DecisionCardinalityShape`), and the *sourceConstraint* (`MaxCountConstraintComponent`). This precision is what makes SHACL reports more actionable than OWL inconsistency messages.

## 3. Deliberate Break #2 — Range Violation

`:CreditScoreRangeShape` restricts FICO to [300, 850]. Let's see what happens when we inject FICO=999.

> 🔧 **Tech**: `sh:datatype` + `sh:minInclusive`/`sh:maxInclusive` range constraints
> 🎯 **Goal**: Inject a non-integer IRI and an out-of-range value to test both datatype and range checks simultaneously
> ✅ **Verify**: `conforms=False` + `DatatypeConstraintComponent` appears in the report
> 📚 **Takeaway**: SHACL validates both data type and numeric range; multiple constraints can be stacked on a single property shape

In [4]:
bad2 = Graph()
for t in data: bad2.add(t)
bad2.remove((CR.Applicant_P01, CR.hasCreditScore, None))  # clear existing
bad2.add((CR.Applicant_P01, CR.hasCreditScore, URIRef("urn:weirdmark") ))  # type mismatch
# Also add an out-of-range legal integer on another applicant
bad2.add((CR.Applicant_S01, CR.hasCreditScore, URIRef("urn:x") ))

conforms, _, report = run(bad2)
print(f"Conforms: {conforms}")
# Show first 1500 characters
print(report[:1500])

Conforms: False
Validation Report
Conforms: False
Results (6):
Constraint Violation in DatatypeConstraintComponent (http://www.w3.org/ns/shacl#DatatypeConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:datatype xsd:integer ; sh:maxInclusive Literal("850", datatype=xsd:integer) ; sh:message Literal("FICO must be in [300, 850]", lang=en) ; sh:minInclusive Literal("300", datatype=xsd:integer) ; sh:path :hasCreditScore ]
	Focus Node: <https://nikko.dev/ontology/credit#Applicant_P01>
	Value Node: <urn:weirdmark>
	Result Path: :hasCreditScore
	Message: FICO must be in [300, 850]
Constraint Violation in DatatypeConstraintComponent (http://www.w3.org/ns/shacl#DatatypeConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:datatype xsd:integer ; sh:maxInclusive Literal("850", datatype=xsd:integer) ; sh:message Literal("FICO must be in [300, 850]", lang=en) ; sh:minInclusive Literal("300", datatype=xsd:integer) ; sh:path :hasCreditScore ]
	Focus Node: <https://nikko.dev/

## 4. Deliberate Break #3 — Property Path

`:ApplicantFicoReachableShape` uses the property path `(:hasApplicant :hasCreditScore)` to require
that every application can reach a FICO value via two hops. Deleting one applicant's FICO
causes every application pointing to that applicant to fail.

> 🔧 **Tech**: `sh:path` composite path (sequence path) + `sh:minCount` cross-node check
> 🎯 **Goal**: Remove P02's FICO so all applications referencing P02 fail the two-hop path check
> ✅ **Verify**: `conforms=False` + multiple application focusNodes violate the `FicoReachable` shape
> 📚 **Takeaway**: SHACL property paths enable cross-node reachability validation — equivalent to SPARQL property paths but declarative

In [5]:
bad3 = Graph()
for t in data: bad3.add(t)
# Delete P02's FICO — App_M02 / App_L02 should now violate the path shape
bad3.remove((CR.Applicant_P02, CR.hasCreditScore, None))

conforms, report_graph, report = run(bad3)
print(f"Conforms: {conforms}")
# Count actual ValidationResult nodes (shape name never appears in message text)
SH_ValidationResult = "http://www.w3.org/ns/shacl#ValidationResult"
path_violations = sum(
    1 for _, _, o in report_graph.triples((None, RDF.type, None))
    if str(o) == SH_ValidationResult
)
print(f"Path violations: {path_violations}")
print(report[:1500])

Conforms: False
Path violations: 2
Validation Report
Conforms: False
Results (2):
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:message Literal("Application's applicant must have a credit score", lang=en) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path ( :hasApplicant :hasCreditScore ) ]
	Focus Node: <https://nikko.dev/ontology/credit#App_L02>
	Result Path: ( :hasApplicant :hasCreditScore )
	Message: Application's applicant must have a credit score
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:message Literal("Application's applicant must have a credit score", lang=en) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path ( :hasApplicant :hasCreditScore ) ]
	Focus Node: <https://nikko.dev/ontology/credit#App_M02>
	Result Path: ( :hasApplicant :hasCreditScore

## 5. sh:or — Logical Or

`:ApplicationEvidenceShape`: each application must have at least one CreditScoreSignal **or** one CollateralSignal.
Let's see what happens when all signals are removed from App_M01.

> 🔧 **Tech**: `sh:or` logically combines multiple shape branches
> 🎯 **Goal**: Remove all `:emittedSignal` triples from App_M01 so both branches of the `or` fail simultaneously
> ✅ **Verify**: `conforms=False` + report contains `ApplicationEvidenceShape` + `OrConstraintComponent`
> 📚 **Takeaway**: `sh:or` reports a violation only when all branches fail; satisfying any one branch is sufficient to pass

In [6]:
bad4 = Graph()
for t in data: bad4.add(t)
bad4.remove((CR.App_M01, CR.emittedSignal, None))

conforms, _, report = run(bad4)
print(f"Conforms: {conforms}")
# Extract Evidence-related violations
for chunk in report.split("\nConstraint Violation"):
    if "Evidence" in chunk:
        print("---");print(chunk[:600])

Conforms: False
---
 in OrConstraintComponent (http://www.w3.org/ns/shacl#OrConstraintComponent):
	Severity: sh:Violation
	Source Shape: :ApplicationEvidenceShape
	Focus Node: <https://nikko.dev/ontology/credit#App_M01>
	Value Node: <https://nikko.dev/ontology/credit#App_M01>
	Message: Application must emit at least one CreditScoreSignal OR CollateralSignal



## 6. sh:not — Logical Not

`:ApplicantTierDisjointnessShape`: an applicant cannot be both Prime and Subprime at the same time.
OWL also declares this disjoint, but SHACL gives a more direct error.

> 🔧 **Tech**: `sh:not` + `sh:class` explicit type check (no DL inference)
> 🎯 **Goal**: Manually assign both PrimeApplicant and SubprimeApplicant types to P01 to trigger the disjointness violation
> ✅ **Verify**: `conforms=False` + `TierDisjointnessShape` + `NotConstraintComponent` in the report
> 📚 **Takeaway**: SHACL `sh:class` only inspects explicit rdf:type triples, not OWL equivalent-class inference — testing disjoint requires running Pellet first then feeding the result to SHACL (pipeline collaboration)

In [7]:
bad5 = Graph()
for t in data: bad5.add(t)
# Force both Prime + Subprime labels onto P01 (FICO 810, naturally Prime; now forcibly also Subprime)
bad5.add((CR.Applicant_P01, RDF.type, CR.SubprimeApplicant))

# Note: SHACL sh:class only checks explicit rdf:type, not OWL DL.
# Because PrimeApplicant is inferred by Pellet (not asserted), SHACL won't see it
# unless we also add it explicitly here.
# Add the Pellet-inferred type explicitly so the shape can fire:
bad5.add((CR.Applicant_P01, RDF.type, CR.PrimeApplicant))

conforms, _, report = run(bad5)
print(f"Conforms: {conforms}")
for chunk in report.split("\nConstraint Violation"):
    if "TierDisjointness" in chunk:
        print("---");print(chunk[:600])

Conforms: False


---
 in NotConstraintComponent (http://www.w3.org/ns/shacl#NotConstraintComponent):
	Severity: sh:Violation
	Source Shape: :ApplicantTierDisjointnessShape
	Focus Node: <https://nikko.dev/ontology/credit#Applicant_P01>
	Value Node: <https://nikko.dev/ontology/credit#Applicant_P01>
	Message: An applicant cannot be classified as both Prime and Subprime



**Key observation**: SHACL does **not** run OWL DL inference — it only sees explicit triples (or at most RDFS expansion).
To test the "PrimeApplicant + SubprimeApplicant" conflict, you must either run Pellet first and feed the materialized
graph to SHACL, or manually add the explicit type triples (as done above). This is the essence of the SHACL + OWL **pipeline**:

```
A-Box → Pellet (infer types) → SHACL (validate types)
```

SHACL does not replace OWL; each covers a distinct stage.

## 7. sh:sparql — Custom Constraint (Signal Freshness)

SHACL Core cannot express constraints like "timestamp must not be older than 90 days". `sh:sparql` embeds
an arbitrary SPARQL SELECT: any rows returned by the SELECT are treated as violations; the `?value`
binding populates the message template.

`:SignalFreshnessShape` checks that `:signalTimestamp` is not more than 90 days in the past.

> 🔧 **Tech**: `sh:sparql` custom SELECT constraint + `advanced=True` to enable it
> 🎯 **Goal**: Change the timestamp on Sig_M01_credit to 2025-12-01 (>90 days ago) and trigger the SPARQL-computed violation
> ✅ **Verify**: `conforms=False` + report contains `SignalFreshnessShape` + Sig_M01 focusNode + `?value` message template substitution
> 📚 **Takeaway**: SHACL Core cannot express time comparisons; `sh:sparql` embeds an arbitrary SELECT — any row returned counts as a violation

In [8]:
# All fixture signal timestamps are 2026-05-*, currently within 90 days -> conforms
# Deliberately move one signal timestamp to 2025-12-01 (more than 5 months ago)
bad6 = Graph()
for t in data: bad6.add(t)
from rdflib import Literal, XSD
bad6.remove((CR.Sig_M01_credit, CR.signalTimestamp, None))
bad6.add((CR.Sig_M01_credit, CR.signalTimestamp, Literal("2025-12-01T00:00:00Z", datatype=XSD.dateTime)))

conforms, _, report = run(bad6)
print(f"Conforms: {conforms}")
for chunk in report.split("\nConstraint Violation"):
    if "Freshness" in chunk or "90 days" in chunk or "Sig_M01" in chunk:
        print("---");print(chunk[:600])

Conforms: False
---
 in SPARQLConstraintComponent (http://www.w3.org/ns/shacl#SPARQLConstraintComponent):
	Severity: sh:Violation
	Source Shape: :SignalFreshnessShape
	Focus Node: <https://nikko.dev/ontology/credit#Sig_M01_credit>
	Value Node: Literal("2025-12-01T00:00:00+00:00" = 2025-12-01 00:00:00+00:00, datatype=xsd:dateTime)
	Source Constraint: [ sh:message Literal("Signal is older than 90 days — refresh before use", lang=en) ; sh:prefixes :OntologyPrefixes ; sh:select Literal("
            SELECT $this ?value WHERE {
                $this :signalTimestamp ?value .
                FILTER (?value < (NOW() - "


## 8. sh:closed — Schema Lock

`:ApplicantClosedShape` declares that applicants may **not** have any properties outside a whitelist.
This is a core data governance tool — it prevents downstream consumers from silently polluting the ontology
with undeclared fields.

> 🔧 **Tech**: `sh:closed true` + `sh:ignoredProperties` schema lock
> 🎯 **Goal**: Inject an undeclared `:favoriteColor` property onto P01 and trigger the closed-shape violation
> ✅ **Verify**: `conforms=False` + report contains `ClosedConstraintComponent` + `favoriteColor` predicate
> 📚 **Takeaway**: `sh:closed` flips the open-world default to closed; the key governance weapon for preventing downstream pollution of the ontology

In [9]:
# Inject an undeclared property onto P01
bad7 = Graph()
for t in data: bad7.add(t)
bad7.add((CR.Applicant_P01, CR.favoriteColor, URIRef("urn:blue")))

conforms, _, report = run(bad7)
print(f"Conforms: {conforms}")
for chunk in report.split("\nConstraint Violation"):
    if "Closed" in chunk or "favoriteColor" in chunk:
        print("---");print(chunk[:500])

Conforms: False
---
 in ClosedConstraintComponent (http://www.w3.org/ns/shacl#ClosedConstraintComponent):
	Severity: sh:Violation
	Source Shape: :ApplicantClosedShape
	Focus Node: <https://nikko.dev/ontology/credit#Applicant_P01>
	Value Node: <urn:blue>
	Result Path: :favoriteColor
	Message: Applicant has a property not in the declared schema (closed shape)



## 9. NearPrime — Revisiting the NB02 Cliffhanger

In NB02 we saw that expressing the `[620, 740)` interval in OWL requires 13 lines and two restrictions.
The equivalent SHACL expression takes 5 lines:

```turtle
:NearPrimeRangeShape a sh:NodeShape ;
    sh:targetClass :Applicant ;
    sh:property [
        sh:path :hasCreditScore ;
        sh:minInclusive 620 ;
        sh:maxExclusive 740 ;
    ] .
```

But the **purpose is completely different**:
- The OWL version is a **definition** — Pellet sees an applicant with FICO=700 and **infers** it is `:NearPrimeApplicant`
- The SHACL version is a **validator** — given an applicant, check whether FICO falls in [620, 740); it does **not** assign a label

If you want to *classify* who is NearPrime, use OWL.
If you want to *guard* against garbage values like FICO=200 entering the store, use SHACL.

## You Should Now Be Able To ✓

- [ ] Distinguish Node Shape from Property Shape
- [ ] Write five constraint types: cardinality / range / property-path / sh:or / sh:not
- [ ] Use `sh:sparql` to embed a custom SELECT as a constraint
- [ ] Read a ValidationReport to extract focusNode + sourceShape + sourceConstraintComponent
- [ ] Use `sh:closed` to lock down a schema
- [ ] Correctly choose OWL vs SHACL when the task is "classify" vs "validate"

## Next Steps

NB04 (SWRL) is the other half of Phase C — applying rules to make business decisions,
and hands-on experience with SWRL's negation-as-failure limitation (the pedagogical pivot behind R5).